# Import dependencies

In [37]:
!python -m pip install --quiet -r ../requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [38]:
import pandas as pd

df = pd.read_csv('../data/processed/telco-churn-cleaned.csv')

X = df.drop(
    columns=['customerID', 'Churn']
)

y = df['Churn']

## 1. Split train/validation/test

### Split 1

In [39]:
from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=69,
    stratify=y,
)

### Split 2

In [40]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.2,
    random_state=67,
    stratify=y_dev,
)

### Kiểm tra

In [41]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(4507, 19)
(1127, 19)
(1409, 19)


In [42]:
print(f"Full: {y.value_counts(normalize=True)}")

print(f"Train: {y_train.value_counts(normalize=True)}")
print(f"Validation: {y_val.value_counts(normalize=True)}")
print(f"Test: {y_test.value_counts(normalize=True)}")

Full: Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64
Train: Churn
No     0.734635
Yes    0.265365
Name: proportion, dtype: float64
Validation: Churn
No     0.734694
Yes    0.265306
Name: proportion, dtype: float64
Test: Churn
No     0.734564
Yes    0.265436
Name: proportion, dtype: float64


## 2. Encoding

In [43]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
binary_features = [
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]
multiclass_features = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

In [44]:
print(len(numerical_features))
print(len(binary_features))
print(len(multiclass_features))

3
5
11


In [45]:
X_train_pre = X_train.copy()
X_val_pre = X_val.copy()
X_test_pre = X_test.copy()

### Xử lý binary

In [46]:
yes_no_mapping = {
    "No": 0,
    "Yes": 1,
}

In [47]:
for col in binary_features:
    if col != 'SeniorCitizen':
        X_train_pre[col] = X_train_pre[col].map(yes_no_mapping)
        X_val_pre[col] = X_val_pre[col].map(yes_no_mapping)
        X_test_pre[col] = X_test_pre[col].map(yes_no_mapping)

In [48]:
for col in binary_features:
    print(
        col,
        X_train_pre[col].unique()
    )

SeniorCitizen [0 1]
Partner [1 0]
Dependents [0 1]
PhoneService [1 0]
PaperlessBilling [1 0]


### Xử lý Multiclass

In [49]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

In [50]:
encoder.fit(X_train_pre[multiclass_features])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [51]:
X_train_multi = encoder.transform(X_train_pre[multiclass_features])
X_val_multi = encoder.transform(X_val_pre[multiclass_features])
X_test_multi = encoder.transform(X_test_pre[multiclass_features])

#### Kiểm tra

In [52]:
print(X_train_multi.shape)
print(X_val_multi.shape)
print(X_test_multi.shape)

(4507, 33)
(1127, 33)
(1409, 33)


#### Lấy tên cột

In [53]:
encoder_features_name = encoder.get_feature_names_out(
    multiclass_features
)

#### Chuyển lại thành DataFrame

In [54]:
X_train_multi = pd.DataFrame(
    X_train_multi,
    columns=encoder_features_name,
    index=X_train_pre.index
)
X_val_multi = pd.DataFrame(
    X_val_multi,
    columns=encoder_features_name,
    index=X_val_pre.index
)
X_test_multi = pd.DataFrame(
    X_test_multi,
    columns=encoder_features_name,
    index=X_test_pre.index
)

### Scale Numerical lại

In [55]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaler.fit(
    X_train_pre[numerical_features]
)

StandardScaler()

In [56]:
X_train_num = scaler.transform(X_train_pre[numerical_features])
X_val_num = scaler.transform(X_val_pre[numerical_features])
X_test_num = scaler.transform(X_test_pre[numerical_features])

#### Chuyển lại thành DataFrame

In [57]:
X_train_num = pd.DataFrame(
    X_train_num,
    columns=numerical_features,
    index=X_train_pre.index
)
X_val_num = pd.DataFrame(
    X_val_num,
    columns=numerical_features,
    index=X_val_pre.index
)
X_test_num = pd.DataFrame(
    X_test_num,
    columns=numerical_features,
    index=X_test_pre.index
)

### Lấy Binary

In [58]:
X_train_bin = X_train_pre[binary_features].copy()
X_val_bin = X_val_pre[binary_features].copy()
X_test_bin = X_test_pre[binary_features].copy()

### Ghép cả 3 vào lại 1 DataFrame

In [59]:
X_train_ready = pd.concat(
    [
        X_train_num,
        X_train_bin,
        X_train_multi,
    ],
    axis=1
)
X_val_ready = pd.concat(
    [
        X_val_num,
        X_val_bin,
        X_val_multi,
    ],
    axis=1
)
X_test_ready = pd.concat(
    [
        X_test_num,
        X_test_bin,
        X_test_multi,
    ],
    axis=1
)

### Kiểm tra

In [60]:
print(X_train_ready.shape)
print(X_val_ready.shape)
print(X_test_ready.shape)

(4507, 41)
(1127, 41)
(1409, 41)


In [61]:
assert list(X_train_ready.columns) == list(X_val_ready.columns)
assert list(X_train_ready.columns) == list(X_test_ready.columns)

assert X_train_ready.isna().sum().sum() == 0
assert X_val_ready.isna().sum().sum() == 0
assert X_test_ready.isna().sum().sum() == 0

In [62]:
print(
    X_train_ready.select_dtypes(
        include=["object"]
    ).columns
)

Index([], dtype='str')


In [63]:
X_train_ready[
    numerical_features
].describe()

,tenure,MonthlyCharges,TotalCharges
count,4.507000e+03,4.507000e+03,4.507000e+03
mean,-5.044901e-17,1.142985e-16,9.301536e-17
std,1.000111e+00,1.000111e+00,1.000111e+00
min,-1.321563e+00,-1.530275e+00,-1.005039e+00
25%,-9.561993e-01,-9.856958e-01,-8.294118e-01
50%,-1.442793e-01,1.800835e-01,-3.907053e-01
75%,9.518127e-01,8.358858e-01,6.548150e-01
max,1.601349e+00,1.778396e+00,2.814396e+00


Tất cả check đều pass, đã hoàn thành preprocessing

## 4. Train lại model

In [64]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
)
model.fit(X_train_ready, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [65]:
y_val_pred = model.predict(
    X_val_ready
)
y_val_prob = model.predict_proba(
    X_val_ready
)

In [66]:
model.classes_

array(['No', 'Yes'], dtype=object)

In [67]:
y_val_prob_yes = y_val_prob[:, 1]

In [68]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
val_accuracy = accuracy_score(
    y_val,
    y_val_pred,
)
val_precision = precision_score(
    y_val,
    y_val_pred,
    pos_label="Yes"
)
val_recall = recall_score(
    y_val,
    y_val_pred,
    pos_label="Yes"
)
val_f1 = f1_score(
    y_val, 
    y_val_pred,
    pos_label="Yes"
)
val_roc_auc = roc_auc_score(
    y_val, 
    y_val_prob_yes,
)
val_ap = average_precision_score(
    y_val,
    y_val_prob_yes,
    pos_label="Yes"
)
val_cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=["No", "Yes"]
)

In [69]:
print(f"Accuracy:          {val_accuracy:.4f}")
print(f"Precision (Yes):   {val_precision:.4f}")
print(f"Recall (Yes):      {val_recall:.4f}")
print(f"F1 (Yes):          {val_f1:.4f}")
print(f"ROC-AUC:           {val_roc_auc:.4f}")
print(f"Average Precision: {val_ap:.4f}")

print("\nConfusion Matrix:")
print(val_cm)

Accuracy:          0.8110
Precision (Yes):   0.6641
Recall (Yes):      0.5819
F1 (Yes):          0.6203
ROC-AUC:           0.8500
Average Precision: 0.6690

Confusion Matrix:
[[740  88]
 [125 174]]


In [70]:
naive_val_accuracy = (
    y_val == "No"
).mean()

print(
    f"Naive accuracy: {naive_val_accuracy:.4f}"
)

Naive accuracy: 0.7347


## 5. Tìm threshold tối ưu

### Sweep toàn bộ threhold

In [73]:
import numpy as np

threshold_results = []

for threshold in np.arange(0.1, 0.91, 0.01):
    y_val_pred_threshold = np.where(
        y_val_prob_yes >= threshold,
        "Yes",
        "No",
    )
    
    threshold_results.append({
        "threshold": threshold,
        "accuracy": accuracy_score(
            y_val, 
            y_val_pred_threshold,
        ),
        "precision": precision_score(
            y_val,
            y_val_pred_threshold,
            pos_label="Yes",
        ),
        "recall": recall_score(
            y_val,
            y_val_pred_threshold,
            pos_label="Yes",
        ),
        "f1": f1_score(
            y_val,
            y_val_pred_threshold,
            pos_label="Yes",
        ),
    })
    
threshold_df = pd.DataFrame(
    threshold_results
)

/Users/haovoxx/Desktop/churn-prediction/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/haovoxx/Desktop/churn-prediction/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/haovoxx/Desktop/churn-prediction/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"

In [74]:
threshold_df.head()

,threshold,accuracy,precision,recall,f1
0,0.10,0.606034,0.398884,0.956522,0.562992
1,0.11,0.613132,0.403112,0.953177,0.566600
2,0.12,0.620231,0.407725,0.953177,0.571142
3,0.13,0.634428,0.417277,0.953177,0.580448
4,0.14,0.640639,0.421131,0.946488,0.582904


### Lựa chọn threshold

Có 3 loại threshold khác nhau dựa trên vấn đề business hiện tại:
1. Cân bằng giữa precision và recall $\rightarrow$ maximize f1
2. Rất sợ bỏ sót Churners $\rightarrow$ recall $\ge$ mức nào đó
3. Biết chính xác chi phí FP và FN $\rightarrow$ minimize business cost

Tuy nhiên, hiện tại chưa có cost thật của business.

#### 1. Muốn cân bằng precision và recall

In [77]:
best_f1_row = threshold_df.loc[threshold_df["f1"].idxmax()]

best_f1_row

threshold    0.370000
accuracy     0.787045
precision    0.575835
recall       0.749164
f1           0.651163
Name: 27, dtype: float64

0.37 là threshold tốt nhất giữa các threshold được quan sát theo tiêu chí maximize F1 trên validation set.

#### 2. Muốn bắt nhiều Churners hơn

Ví dụ business nhận định:
- “Phải phát hiện ít nhất 70% churners.”

Tức là
$$ \text{Recall} \ge 0.7$$

In [76]:
high_recall = threshold_df.loc[
    threshold_df["recall"] >= 0.70
]
high_recall

,threshold,accuracy,precision,recall,f1
0,0.10,0.606034,0.398884,0.956522,0.562992
1,0.11,0.613132,0.403112,0.953177,0.566600
2,0.12,0.620231,0.407725,0.953177,0.571142
3,0.13,0.634428,0.417277,0.953177,0.580448
4,0.14,0.640639,0.421131,0.946488,0.582904
5,0.15,0.660160,0.434783,0.936455,0.593849
6,0.16,0.673469,0.444800,0.929766,0.601732
7,0.17,0.681455,0.450980,0.923077,0.605928
8,0.18,0.696539,0.463373,0.909699,0.613995
9,0.19,0.704525,0.470383,0.903010,0.618557


Trong những cái threshold mà có recall đạt yêu cầu đã lọc, chọn những cái có precision cao nhất

In [78]:
best_recall_row = high_recall.loc[
    high_recall["precision"].idxmax()
]
best_recall_row

threshold    0.390000
accuracy     0.787045
precision    0.580822
recall       0.709030
f1           0.638554
Name: 29, dtype: float64

0.39 là threshold tốt nhất giữa các threshold được quan sát theo constraint Recall $\ge$ 0.70 rồi maximize Precision.

### Chọn threshold
Do hiện tại chưa có business cost cụ thể cho False Positive và False Negative,
threshold được chọn theo tiêu chí maximize F1-score trên validation set.

Trong các threshold được khảo sát:

- threshold = 0.37
- Precision = 0.5758
- Recall = 0.7492
- F1 = 0.6512

Do đó, chọn `0.37` làm decision threshold cho baseline hiện tại.

Lưu ý: đây là threshold tốt nhất theo F1 trên validation set,
không phải threshold tốt nhất trong mọi business objective.

In [79]:
selected_threshold = 0.37

## 6. Predict với threshold đã chọn

In [80]:
y_test_prob_yes = model.predict_proba(
    X_test_ready
)[:, 1]

y_test_pred = np.where(
    y_test_prob_yes >= selected_threshold,
    "Yes",
    "No",
)

### Đánh giá

In [81]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred,
)

test_precision = precision_score(
    y_test,
    y_test_pred,
    pos_label="Yes",
)

test_recall = recall_score(
    y_test,
    y_test_pred,
    pos_label="Yes",
)

test_f1 = f1_score(
    y_test,
    y_test_pred,
    pos_label="Yes",
)

test_roc_auc = roc_auc_score(
    y_test,
    y_test_prob_yes,
)

test_ap = average_precision_score(
    y_test,
    y_test_prob_yes,
    pos_label="Yes",
)

test_cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=["No", "Yes"],
)

In [82]:
print(f"Selected threshold: {selected_threshold}")

print(f"Accuracy:          {test_accuracy:.4f}")
print(f"Precision (Yes):   {test_precision:.4f}")
print(f"Recall (Yes):      {test_recall:.4f}")
print(f"F1 (Yes):          {test_f1:.4f}")
print(f"ROC-AUC:           {test_roc_auc:.4f}")
print(f"Average Precision: {test_ap:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

Selected threshold: 0.37
Accuracy:          0.7857
Precision (Yes):   0.5800
Recall (Yes):      0.6979
F1 (Yes):          0.6335
ROC-AUC:           0.8468
Average Precision: 0.6584

Confusion Matrix:
[[846 189]
 [113 261]]


### Final Test Evaluation

Decision threshold được chọn trên validation set theo tiêu chí maximize F1:

\[
threshold = 0.37
\]

Sau khi threshold đã được cố định, model được đánh giá một lần trên test set.

#### Test results

- Accuracy: 0.7857
- Precision (Yes): 0.5800
- Recall (Yes): 0.6979
- F1-score (Yes): 0.6335
- ROC-AUC: 0.8468
- Average Precision: 0.6584

Confusion Matrix:

- TN = 846
- FP = 189
- FN = 113
- TP = 261

#### Nhận xét

Threshold 0.37 được chọn trên validation set, không phải test set.

Trên validation, Recall đạt khoảng 74.9%, nhưng trên unseen test set chỉ còn khoảng
69.8%. Đây là sự khác biệt bình thường khi model generalize sang dữ liệu chưa dùng
để lựa chọn threshold.

Không điều chỉnh lại threshold dựa trên test result, vì điều đó sẽ làm test set
tham gia vào quá trình tuning.

ROC-AUC giảm nhẹ từ 0.8500 trên validation xuống 0.8468 trên test, và Average
Precision giảm từ 0.6690 xuống 0.6584. Hai metric ranking khá ổn định giữa
validation và test.

Kết luận: Logistic Regression baseline có khả năng phân biệt churner tương đối
ổn định, nhưng việc tăng Recall thông qua threshold thấp phải đánh đổi bằng nhiều
False Positive hơn.